# Random Forest Baseline — Drug-Drug Interaction PRR Regression & Significance Classification

This notebook trains a multi-output **Random Forest Regressor** to predict the per-category **PRR_max** signal strength of a drug-drug interaction from ChemBERTa molecular embeddings, then thresholds the predictions to flag **clinically significant** interactions.

It serves as a **baseline model** to benchmark against the GNN.

**Pipeline:**
1. Load ChemBERTa embeddings + TWOSIDES edge labels
2. Pivot labels — one row per drug pair, one PRR_max column per reaction category
3. Build drug pair feature vectors (drug1 ⊕ drug2 embeddings)
4. Train / validation / test split (80 / 10 / 10)
5. Train the multi-output Random Forest
6. Regression metrics (RMSE | MAE)
7. Classification metrics (Precision / Recall / F1) at a PRR threshold
8. Figures + results summary

**Success criterion:** Test Macro F1 > 0.75 (the GNN must exceed this).

**Input files needed:**
- `data/chembert_embeddings.csv`
- `data/twosides_edge_labels.csv`


## Imports & Settings

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# ── Settings ──────────────────────────────────────────
DATA_DIR     = Path("data")        # folder with the two input CSVs
OUT_DIR      = Path("outputs")     # folder to write figures + CSVs into
N_TREES      = 300                 # number of trees in the forest
MIN_LEAF     = 2                   # min_samples_leaf for each tree
PRR_THRESH   = 2.0                 # PRR cutoff for "clinically significant"
SAMPLE_SIZE  = None                # set to an int to subsample drug pairs
RANDOM_STATE = 42
# ───────────────────────────────────────────────

OUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(RANDOM_STATE)

EMB_PATH    = DATA_DIR / "chembert_embeddings.csv"
LABELS_PATH = DATA_DIR / "twosides_edge_labels.csv"

for p in (EMB_PATH, LABELS_PATH):
    if not p.exists():
        sys.exit(f"ERROR: required input file not found: {p}")

# Lightweight elapsed-time logger used throughout the notebook
t0 = time.time()
def stamp(msg):
    print(f"[{time.time()-t0:6.1f}s] {msg}")

print("All imports successful")
print(f"Trees: {N_TREES} | min_leaf: {MIN_LEAF} | PRR threshold: {PRR_THRESH}")


## Step 1 — Load Data

Load the ChemBERTa embeddings (one row per drug) and the TWOSIDES edge labels (one row per drug-pair / reaction-category).

In [ ]:
print("=" * 60)
print("1. Loading data")
print("=" * 60)

emb_df  = pd.read_csv(EMB_PATH, index_col="drugbank_id")
emb_dim = emb_df.shape[1]
stamp(f"Embedding dimension per drug : {emb_dim}")
stamp(f"Drugs with embeddings        : {len(emb_df):,}")

label_df = pd.read_csv(LABELS_PATH)
expected = {"drug_1_drugbank_id", "drug_2_drugbank_id", "reaction_category", "PRR_max"}
missing  = expected - set(label_df.columns)
if missing:
    sys.exit(f"ERROR: {LABELS_PATH} missing columns: {missing}")

stamp(f"DDI rows (raw)               : {len(label_df):,}")
stamp(f"Unique reaction categories   : {label_df['reaction_category'].nunique()}")
print(f"   Categories: {sorted(label_df['reaction_category'].unique())}")


## Step 2 — Pivot Labels

Reshape the long edge-label table into one row per drug pair, with one `PRR_max` column per reaction category. Missing pair/category combinations are filled with 0 (no signal).

In [ ]:
print("\n" + "=" * 60)
print("2. Pivoting labels")
print("=" * 60)

pivot_df = (
    label_df
    .pivot_table(
        index=["drug_1_drugbank_id", "drug_2_drugbank_id"],
        columns="reaction_category",
        values="PRR_max",
        aggfunc="max",
    )
    .fillna(0)
    .reset_index()
)
pivot_df.columns.name = None

reaction_cols = [
    c for c in pivot_df.columns
    if c not in ("drug_1_drugbank_id", "drug_2_drugbank_id")
]

stamp(f"Unique drug pairs : {len(pivot_df):,}")
stamp(f"Reaction categories ({len(reaction_cols)}): {reaction_cols}")


## Step 3 — Build Feature Matrix

For each drug pair, concatenate the ChemBERTa embedding of drug 1 and drug 2 into a single feature vector. Pairs where either drug has no embedding are dropped. Optionally subsample to `SAMPLE_SIZE` pairs.

In [ ]:
print("\n" + "=" * 60)
print("3. Building feature matrix")
print("=" * 60)

known_ids = set(emb_df.index)
mask = (
    pivot_df["drug_1_drugbank_id"].isin(known_ids) &
    pivot_df["drug_2_drugbank_id"].isin(known_ids)
)
dropped  = (~mask).sum()
pivot_df = pivot_df[mask].reset_index(drop=True)
stamp(f"Pairs retained : {len(pivot_df):,}  (dropped {dropped:,} missing embeddings)")

if SAMPLE_SIZE is not None and len(pivot_df) > SAMPLE_SIZE:
    pivot_df = pivot_df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
    stamp(f"Subsampled to  : {len(pivot_df):,} pairs (SAMPLE_SIZE={SAMPLE_SIZE})")

emb1 = emb_df.loc[pivot_df["drug_1_drugbank_id"]].values
emb2 = emb_df.loc[pivot_df["drug_2_drugbank_id"]].values

X = np.hstack([emb1, emb2]).astype(np.float32)
Y = pivot_df[reaction_cols].values.astype(np.float32)

stamp(f"Feature matrix X : {X.shape}  ({X.nbytes / 1e9:.2f} GB)")
stamp(f"Target matrix  Y : {Y.shape}")
stamp(f"PRR_max range    : {Y.min():.2f} - {Y.max():.2f}")


## Step 4 — Train / Validation / Test Split

80 / 10 / 10 split. We first hold out 10% as test, then carve 1/9 of the remaining 90% (= 10% of the original) as validation.

In [ ]:
print("\n" + "=" * 60)
print("4. Splitting data  80 / 10 / 10")
print("=" * 60)

X_trainval, X_test, Y_trainval, Y_test = train_test_split(
    X, Y, test_size=0.10, random_state=RANDOM_STATE
)
# 1/9 of the remaining 90% == 10% of the original total
X_train, X_val, Y_train, Y_val = train_test_split(
    X_trainval, Y_trainval,
    test_size=1/9,
    random_state=RANDOM_STATE,
)

stamp(f"Train samples      : {len(X_train):,}")
stamp(f"Validation samples : {len(X_val):,}")
stamp(f"Test samples       : {len(X_test):,}")


## Step 5 — Train Random Forest

A multi-output `RandomForestRegressor` predicts the `PRR_max` for every reaction category at once.

Key settings:
- `n_estimators=300` — number of trees
- `min_samples_leaf=2` — limits overfitting on individual leaves
- `max_features="sqrt"` — random feature subset per split (decorrelates trees)
- `oob_score=True` — out-of-bag R² as a free validation estimate
- `n_jobs=-1` — train across all CPU cores

In [ ]:
print("\n" + "=" * 60)
print(f"5. Training Random Forest  ({N_TREES} trees, {len(reaction_cols)} outputs)")
print("=" * 60)

rf = RandomForestRegressor(
    n_estimators=N_TREES,
    min_samples_leaf=MIN_LEAF,
    max_features="sqrt",
    n_jobs=-1,
    oob_score=True,
    bootstrap=True,   # required for oob_score
    random_state=RANDOM_STATE,
)
rf.fit(X_train, Y_train)
stamp(f"Done.  OOB R2 score: {rf.oob_score_:.4f}")

# Predict (clip negative PRR predictions to 0)
Y_val_pred  = rf.predict(X_val).clip(0, None)
Y_test_pred = rf.predict(X_test).clip(0, None)


## Step 6 — Regression Metrics (RMSE | MAE)

Per-category root-mean-squared error and mean-absolute error on the predicted `PRR_max`, for both the validation and test sets.

In [ ]:
print("\n" + "=" * 60)
print("6. Regression Metrics  (RMSE | MAE)")
print("=" * 60)
print(
    f"{'Category':<20} {'Val RMSE':>10} {'Val MAE':>10} |"
    f" {'Test RMSE':>10} {'Test MAE':>10}"
)
print("-" * 67)

val_rmses, val_maes, test_rmses, test_maes = [], [], [], []
for i, cat in enumerate(reaction_cols):
    vr = float(np.sqrt(mean_squared_error(Y_val[:, i],  Y_val_pred[:, i])))
    vm = float(mean_absolute_error(Y_val[:, i],  Y_val_pred[:, i]))
    tr = float(np.sqrt(mean_squared_error(Y_test[:, i], Y_test_pred[:, i])))
    tm = float(mean_absolute_error(Y_test[:, i], Y_test_pred[:, i]))
    val_rmses.append(vr); val_maes.append(vm)
    test_rmses.append(tr); test_maes.append(tm)
    print(f"{cat:<20} {vr:>10.4f} {vm:>10.4f} | {tr:>10.4f} {tm:>10.4f}")

print("-" * 67)
print(
    f"{'MEAN':<20} {np.mean(val_rmses):>10.4f} {np.mean(val_maes):>10.4f} |"
    f" {np.mean(test_rmses):>10.4f} {np.mean(test_maes):>10.4f}"
)


## Step 7 — Classification Metrics (Precision / Recall / F1)

We binarise both the true and predicted `PRR_max` at `PRR_THRESH` to define a "clinically significant" interaction, then compute precision, recall, and F1 per category.

**Success criterion:** Macro F1 > 0.75 on the test set.

In [ ]:
print("\n" + "=" * 60)
print(f"7. Classification Metrics  (PRR threshold = {PRR_THRESH})")
print("=" * 60)

Y_val_bin  = (Y_val       >= PRR_THRESH).astype(int)
Y_test_bin = (Y_test      >= PRR_THRESH).astype(int)
P_val_bin  = (Y_val_pred  >= PRR_THRESH).astype(int)
P_test_bin = (Y_test_pred >= PRR_THRESH).astype(int)

print(
    f"{'Category':<20} {'V-Prec':>8} {'V-Rec':>8} {'V-F1':>8} |"
    f" {'T-Prec':>8} {'T-Rec':>8} {'T-F1':>8}"
)
print("-" * 79)

per_cat_rows = []
val_f1s, test_f1s = [], []
for i, cat in enumerate(reaction_cols):
    vp = precision_score(Y_val_bin[:, i],  P_val_bin[:, i],  zero_division=0)
    vr = recall_score(   Y_val_bin[:, i],  P_val_bin[:, i],  zero_division=0)
    vf = f1_score(       Y_val_bin[:, i],  P_val_bin[:, i],  zero_division=0)
    tp = precision_score(Y_test_bin[:, i], P_test_bin[:, i], zero_division=0)
    tr = recall_score(   Y_test_bin[:, i], P_test_bin[:, i], zero_division=0)
    tf = f1_score(       Y_test_bin[:, i], P_test_bin[:, i], zero_division=0)
    val_f1s.append(vf); test_f1s.append(tf)
    print(f"{cat:<20} {vp:>8.4f} {vr:>8.4f} {vf:>8.4f} | {tp:>8.4f} {tr:>8.4f} {tf:>8.4f}")

    per_cat_rows.append({
        "category":  cat,
        "val_rmse":  val_rmses[i],
        "val_mae":   val_maes[i],
        "test_rmse": test_rmses[i],
        "test_mae":  test_maes[i],
        "val_prec":  vp, "val_rec":  vr, "val_f1":  vf,
        "test_prec": tp, "test_rec": tr, "test_f1": tf,
        "n_test_positives": int(Y_test_bin[:, i].sum()),
    })

macro_val_f1  = float(np.mean(val_f1s))
macro_test_f1 = float(np.mean(test_f1s))
print("-" * 79)
print(
    f"{'MACRO AVG':<20} {'':>8} {'':>8} {macro_val_f1:>8.4f} |"
    f" {'':>8} {'':>8} {macro_test_f1:>8.4f}"
)

print()
print("=" * 55)
status = "PASSED" if macro_test_f1 > 0.75 else "NOT MET -- GNN must exceed this"
print(f"SUCCESS CRITERION (Test Macro F1 > 0.75): {status}")
print(f"Test Macro F1 = {macro_test_f1:.4f}")
print("=" * 55)


## Step 8 — Figures

Four diagnostic figures are written to the `outputs/` folder:
1. Predicted vs actual PRR scatter (test set)
2. Confusion matrices (validation & test, all categories combined)
3. Per-category F1 bar chart vs the 0.75 success line
4. Top-30 feature importances

In [ ]:
print("\nGenerating figures...")

# 8a. Predicted vs Actual PRR
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(Y_test.flatten(), Y_test_pred.flatten(),
           alpha=0.15, s=4, color="steelblue")
max_val = float(max(Y_test.max(), Y_test_pred.max()))
ax.plot([0, max_val], [0, max_val], "r--", linewidth=1.5, label="Perfect prediction")
ax.set_xlabel("Actual PRR_max")
ax.set_ylabel("Predicted PRR_max")
ax.set_title("Predicted vs Actual PRR — Test Set (all categories)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "rf_pred_vs_actual.png", dpi=150)
plt.show()
stamp(f"Saved {OUT_DIR / 'rf_pred_vs_actual.png'}")


In [ ]:
# 8b. Confusion matrices (validation & test)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, Y_true_b, P_pred_b, split in zip(
        axes,
        [Y_val_bin,  Y_test_bin],
        [P_val_bin,  P_test_bin],
        ["Validation", "Test"]):
    cm = confusion_matrix(Y_true_b.flatten(), P_pred_b.flatten())
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No signal", f"Significant (PRR>={PRR_THRESH})"],
    )
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(
        f"Confusion Matrix — {split} Set\n"
        f"(PRR threshold={PRR_THRESH}, all categories combined)",
        fontsize=10,
    )
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig(OUT_DIR / "rf_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
stamp(f"Saved {OUT_DIR / 'rf_confusion_matrix.png'}")


In [ ]:
# 8c. Per-category F1 bar chart
x = np.arange(len(reaction_cols))
width = 0.35

fig, ax = plt.subplots(figsize=(max(8, len(reaction_cols) * 1.2), 5))
ax.bar(x - width/2, val_f1s,  width, label="Validation", color="steelblue", alpha=0.85)
ax.bar(x + width/2, test_f1s, width, label="Test",       color="tomato",    alpha=0.85)
ax.axhline(0.75, color="black", linestyle="--", linewidth=1.2,
           label="Success criterion (F1=0.75)")
ax.set_xticks(x)
ax.set_xticklabels(reaction_cols, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("F1 Score")
ax.set_ylim(0, 1.05)
ax.set_title("Per-Category F1 Score — Random Forest Baseline")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "rf_f1_per_category.png", dpi=150, bbox_inches="tight")
plt.show()
stamp(f"Saved {OUT_DIR / 'rf_f1_per_category.png'}")


In [ ]:
# 8d. Top 30 feature importances
importances = rf.feature_importances_
feat_names = (
    [f"drug1_d{i}" for i in range(emb_dim)] +
    [f"drug2_d{i}" for i in range(emb_dim)]
)

top_n     = 30
top_idx   = np.argsort(importances)[::-1][:top_n]
top_vals  = importances[top_idx]
top_names = [feat_names[i] for i in top_idx]
colors    = ["#4e79a7" if "drug1" in n else "#f28e2b" for n in top_names]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(top_n), top_vals, color=colors)
ax.set_xticks(range(top_n))
ax.set_xticklabels(top_names, rotation=90, fontsize=7)
ax.set_ylabel("Mean Decrease in Impurity")
ax.set_title("Top 30 Feature Importances  |  blue = drug1  |  orange = drug2")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "rf_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
stamp(f"Saved {OUT_DIR / 'rf_feature_importance.png'}")


## Step 9 — Results Summary

Aggregate metrics are written to `outputs/rf_results_summary.csv` and the full per-category breakdown to `outputs/rf_per_category_metrics.csv`.

In [ ]:
summary = pd.DataFrame({
    "Split":       ["Validation", "Test"],
    "RMSE (mean)": [float(np.mean(val_rmses)), float(np.mean(test_rmses))],
    "MAE (mean)":  [float(np.mean(val_maes)),  float(np.mean(test_maes))],
    "Macro F1":    [macro_val_f1,              macro_test_f1],
}).set_index("Split").round(4)

summary.to_csv(OUT_DIR / "rf_results_summary.csv")
pd.DataFrame(per_cat_rows).round(4).to_csv(OUT_DIR / "rf_per_category_metrics.csv", index=False)

print("=" * 60)
print("Results Summary")
print("=" * 60)
display(summary)

print(f"\nOOB R2: {rf.oob_score_:.4f}")
print(f"\nSuccess criterion (Test Macro F1 > 0.75): {status}")
print(f"\nAll outputs saved to {OUT_DIR}/")
print(f"Total runtime: {time.time()-t0:.1f}s")
print("Done.")
print(f"\nGNN target: F1 > 0.75  (Random Forest = {macro_test_f1:.4f})")
